# Lumped Element Model Design

### Specs
Order: 5th Chebyshev

f0 = 30GHz

BW: 2GHz

Fractional Bandwidth: 13.33%


##### This is for the lowpass filter prototype -> before translated to a bandpass filter

Sources Used:

Cameron :
Anatol I. Zverev Handbook of filter synthesis: https://ia803101.us.archive.org/20/items/HandbookOfFilterSynthesis/Handbook%20of%20Filter%20Synthesis.pdf
Specifically this line comes from : beta = np.log(1/(np.tanh(L_AR/2*20*np.log(2))))= np.log(1/(np.tanh(L_AR/17.37))) https://eng.libretexts.org/Bookshelves/Electrical_Engineering/Electronics/Microwave_and_RF_Design_IV%3A_Modules_(Steer)/02%3A_Filters/2.07%3A_Butterworth_and_Chebyshev_Filters

In [69]:
import numpy as np

In [70]:
# Constants
#  Ripple level chosen to be high to increase the selectivity of the filter
L_AR = 0.1 # 0.5 # dB ripple level
N = 5      # Filter Order
e = np.sqrt(10**(L_AR/10) - 1)
ref_coef = e/np.sqrt(1+e**2)
RL = -20*np.log10(ref_coef)
print("RL =", RL)
del_BW = 4/30
f0 = 30*10**(9)
w0 = 2*np.pi*f0
Z0 = 50
g = np.zeros(N+2)
a = np.zeros(N)
b = np.zeros(N)
g_bpf = np.zeros(2*N)
M = np.zeros((N+2, N+2))


RL = 16.427747172383718


In [71]:
# Calculating values of g-parameters (lowpass prototype) using formulae found in Cameron, Matthaei
# Values verified through Table given in Matthaei, Young, Jones, Microwave Filters, Impedance-Matching Networks, and Coupling Structures
# Table 4.05-2a

g[0] = 1
beta = np.log(1/(np.tanh(L_AR/17.37)))
gamma = np.sinh(beta/(2*N))

for k in range(1, N+1, 1):
    a[k-1] = np.sin((np.pi*(2*k-1))/(2*N))

for m in range(1, N+1, 1):
    b[m-1] = gamma**2 + (np.sin((m*np.pi)/(N)))**2

for i in range(1, N+2, 1):
    if i == 1:
        g[i] = (2*a[0])/(gamma)
    if i == N+1:
        g[i] = 1
    if 1<i<N+1: 
        g[i] = (4*a[i-2]*a[i-1])/(b[i-2]*g[i-1])

print(g)

[1.         1.14683783 1.37120999 1.97502758 1.37120999 1.14683783
 1.        ]


In [72]:
# Lowpass to Bandpass transform with frequency scaling to w0

for i in range(len(g)):
    print("g[{}] = {}".format(i, g[i]))
print("=============")
# #  Given g1 is an inductor
# g_bpf[0] = (Z0*g[1])/(del_BW*w0) # L1
# g_bpf[1] = del_BW/(w0*g[1]*Z0) # C1

# g_bpf[2] = Z0*del_BW/(w0*g[2]) # L2
# g_bpf[3] = (g[2])/(del_BW*w0*Z0) # C2

# g_bpf[4] = (Z0*g[3])/(del_BW*w0) # L3
# g_bpf[5] = del_BW/(w0*g[3]*Z0) # C3

# print("L1' =", g_bpf[0]*10**(9), "nH")
# print("C1' =", g_bpf[1]*10**(12), "pF")
# print("L2' =", g_bpf[2]*10**(9), "nH")
# print("C2' =", g_bpf[3]*10**(12), "pF")
# print("L3' =", g_bpf[4]*10**(9), "nH")
# print("C3' =", g_bpf[5]*10**(12), "pF")

# print("=============")

for i in range(N):
    if i%2 == 0:
        g_bpf[2*i] = (Z0*g[i+1])/(del_BW*w0) # L
        g_bpf[2*i+1] = del_BW/(w0*g[i+1]*Z0) # C
    else:
        g_bpf[2*i] = Z0*del_BW/(w0*g[i+1]) # L
        g_bpf[2*i+1] = (g[i+1])/(del_BW*w0*Z0) # C

print("=============")

print("L1' =", g_bpf[0]*10**(9), "nH")
print("C1' =", g_bpf[1]*10**(12), "pF")
print("L2' =", g_bpf[2]*10**(9), "nH")
print("C2' =", g_bpf[3]*10**(12), "pF")
print("L3' =", g_bpf[4]*10**(9), "nH")
print("C3' =", g_bpf[5]*10**(12), "pF")
print("L4' =", g_bpf[6]*10**(9), "nH")
print("C4' =", g_bpf[7]*10**(12), "pF")
print("L5' =", g_bpf[8]*10**(9), "nH")
print("C5' =", g_bpf[9]*10**(12), "pF") 


g[0] = 1.0
g[1] = 1.1468378278006446
g[2] = 1.371209987510143
g[3] = 1.975027577855714
g[4] = 1.3712099875101427
g[5] = 1.1468378278006448
g[6] = 1.0
L1' = 2.281561365240556 nH
C1' = 0.012335751149527063 pF
L2' = 0.025793106419647258 nH
C2' = 1.091174237646077 pF
L3' = 3.929192521981876 nH
C3' = 0.007162991651981094 pF
L4' = 0.025793106419647265 nH
C4' = 1.0911742376460767 pF
L5' = 2.2815613652405564 nH
C5' = 0.012335751149527058 pF


In [73]:
#  Find the coupling matrix of the filter

for i in range(N+1):
    if i == 0:
        # M[i][i+1] = 1/np.sqrt(g[0]*g[1]) # book says no square root here but if you dont then it doesnt match the cst filter designer 3d
        # M[i+1][i] = 1/np.sqrt(g[0]*g[1])
        M[i][i+1] = 1/(g[0]*g[1]) # book says no square root here but if you dont then it doesnt match the cst filter designer 3d
        M[i+1][i] = 1/(g[0]*g[1])
    if i == N:
        M[i][i+1] = 1/np.sqrt(g[N]*g[N+1]) # same here
        M[i+1][i] = 1/np.sqrt(g[N]*g[N+1])
    if 1<=i<N:
        M[i][i+1] = 1/np.sqrt(g[2]*g[3])
        M[i+1][i] = 1/np.sqrt(g[2]*g[3])

print(M)

[[0.         0.87196287 0.         0.         0.         0.
  0.        ]
 [0.87196287 0.         0.6076611  0.         0.         0.
  0.        ]
 [0.         0.6076611  0.         0.6076611  0.         0.
  0.        ]
 [0.         0.         0.6076611  0.         0.6076611  0.
  0.        ]
 [0.         0.         0.         0.6076611  0.         0.6076611
  0.        ]
 [0.         0.         0.         0.         0.6076611  0.
  0.93378952]
 [0.         0.         0.         0.         0.         0.93378952
  0.        ]]


In [74]:
#  Calculate the Q-value of the end resonators including the bandwidth scaling del_BW

q_S = (1/del_BW)*(g[1]/g[0])
q_L = (1/del_BW)*(g[3]*g[N]/g[N+1])

print("q_S =", q_S, "and q_L =", q_L)

q_S = 8.601283708504834 and q_L = 16.98777252925812


In [75]:
# Calculate the coupling coefficients (This case is for when all resonators are equal)

k = del_BW / np.sqrt(g[1:N] * g[2:N+1])
for i in range(len(k)):
    print("k_{}{} = {}".format(i+1, i+2, k[i]))   

k_12 = 0.10632508729198305
k_23 = 0.08102147972210084
k_34 = 0.08102147972210086
k_45 = 0.10632508729198305


In [82]:
L_0 = 330*10**(-12) # 360p
L_1 = g[1] # the first inductor value in the bandpass ladder circuit
R = L_0*(del_BW*w0)/L_1      # This is the impedance scaling factor -> it is not a real resistor 
C_0 = 1/(L_0*w0**2)

print("R =", R)
print("L_0 =", L_0*10**(12), "pH")
print("C_0 =", C_0*10**(12), "pF")  

R = 7.231889639865254
L_0 = 330.0 pH
C_0 = 0.08528719161812945 pF


In [77]:
# Find the K Matrix
K = k*w0*L_0

for i in range(len(K)):
    print("K_{}{} = {}".format(i+1, i+2, K[i]))


K_12 = 6.613796239949989
K_23 = 5.039822412462942
K_34 = 5.039822412462944
K_45 = 6.613796239949989


In [78]:
#  Find actual circuit values for the shunt J inverter (ie the pi capaicor topology)
#  K_ij = wL_ij
L = K/w0
for i in range(len(L)):
    print("L_{}{} = {} pH".format(i+1, i+2, L[i]*10**12))

L_12 = 35.0872788063544 pH
L_23 = 26.737088308293277 pH
L_34 = 26.737088308293284 pH
L_45 = 35.0872788063544 pH
